# Lecture 8: Cross-Validation for Lasso, Ridge, and Elastic Net

### Short, simple, self-study notes

Regularized models need hyperparameters. Cross-validation helps choose them using training data without touching the final test set.

**Main idea:** Try several settings on training folds, choose the setting with the best average validation result, then evaluate once on the test set.

## 1. What does cross-validation choose?

- **LassoCV** chooses alpha, the L1 penalty strength.
- **RidgeCV** chooses alpha, the L2 penalty strength.
- **ElasticNetCV** chooses alpha and can compare L1/L2 mixing values with l1_ratio.

With 5-fold CV, the training data is divided into five parts. Each part becomes validation data once, while the remaining four parts are used for fitting. The average validation result is used to choose hyperparameters.

The final test set is not used to choose alpha.

## 2. The safe workflow

1. Split the original dataset into training and test data.
2. Scale features using training data only.
3. Run CV inside the training data to select alpha and other settings.
4. Fit the chosen model on the training data.
5. Predict the untouched test data once.

Putting scaling and modelling in a pipeline is the safest approach in a real project. This notebook scales before CV for clarity, because it continues the earlier lessons.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data_path = Path("Ridge Lassso Elastic Regression Practicals") / "Algerian_forest_fires_cleaned_dataset.csv"
df = pd.read_csv(data_path)
df["Classes"] = df["Classes"].str.strip().str.lower().map({"not fire": 0, "fire": 1})

X = df.drop(columns=["day", "month", "year", "FWI"])
y = df["FWI"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training rows used for CV:", len(X_train))
print("Final test rows kept aside:", len(X_test))

## 3. LassoCV

LassoCV fits Lasso models for many alpha values. It measures validation error across folds and selects the alpha with the lowest average validation error.

Useful fitted attributes:

- alpha_: the selected penalty strength.
- alphas_: all alpha values tried.
- mse_path_: validation mean-squared-error values for the alpha path and folds.
- coef_: the final coefficients; some can be zero.

In [ ]:
lasso_cv = LassoCV(cv=5, max_iter=20_000, random_state=42)
lasso_cv.fit(X_train_scaled, y_train)
lasso_prediction = lasso_cv.predict(X_test_scaled)

print("Selected Lasso alpha:", lasso_cv.alpha_)
print("Number of alphas tried:", len(lasso_cv.alphas_))
print("Test MAE:", round(mean_absolute_error(y_test, lasso_prediction), 3))
print("Test R2:", round(r2_score(y_test, lasso_prediction), 3))
print("Zero coefficients:", int(np.sum(np.isclose(lasso_cv.coef_, 0))))

## 4. Visual: Lasso alpha versus validation error

This diagram averages the validation MSE across the five folds for every alpha tried.

- Lower MSE is better.
- The vertical line is the alpha selected by LassoCV.
- Very small alpha behaves closer to ordinary linear regression.
- Very large alpha can make the model too simple.

In [ ]:
mean_mse = lasso_cv.mse_path_.mean(axis=1)

plt.figure(figsize=(9, 4))
plt.semilogx(lasso_cv.alphas_, mean_mse, marker="o", markersize=3, color="#457b9d")
plt.axvline(lasso_cv.alpha_, color="#e76f51", linestyle="--", label=f"selected alpha = {lasso_cv.alpha_:.4g}")
plt.gca().invert_xaxis()
plt.xlabel("alpha (stronger L1 penalty to the left)")
plt.ylabel("average validation MSE")
plt.title("LassoCV chooses alpha with low cross-validation error", weight="bold")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

## 5. RidgeCV and ElasticNetCV

RidgeCV tries the alpha values we provide. ElasticNetCV tries alpha values and compares the given L1 ratios.

For Elastic Net:

- l1_ratio close to 1 means more Lasso-like behavior.
- l1_ratio close to 0 means more Ridge-like behavior.
- A middle value mixes both penalties.

RidgeCV with cv=None uses an efficient generalized cross-validation method. In this lesson, cv=5 is set explicitly so all three models use five folds.

In [ ]:
ridge_alphas = np.logspace(-3, 3, 40)
ridge_cv = RidgeCV(alphas=ridge_alphas, cv=5, scoring="neg_mean_absolute_error")
ridge_cv.fit(X_train_scaled, y_train)
ridge_prediction = ridge_cv.predict(X_test_scaled)

elastic_cv = ElasticNetCV(
    l1_ratio=[0.1, 0.5, 0.9], cv=5, max_iter=20_000, random_state=42
)
elastic_cv.fit(X_train_scaled, y_train)
elastic_prediction = elastic_cv.predict(X_test_scaled)

print("Selected Ridge alpha:", ridge_cv.alpha_)
print("Selected Elastic Net alpha:", elastic_cv.alpha_)
print("Selected Elastic Net l1_ratio:", elastic_cv.l1_ratio_)

## 6. Compare tuned models on the test set

After CV chooses the hyperparameters using the training data, we compare the three tuned models once on the test set.

The test set measures final performance. Do not repeatedly change settings after looking at this result, or it stops being a fair test.

In [ ]:
tuned_predictions = {
    "LassoCV": lasso_prediction,
    "RidgeCV": ridge_prediction,
    "ElasticNetCV": elastic_prediction,
}

comparison = pd.DataFrame({
    name: {
        "test_MAE": mean_absolute_error(y_test, prediction),
        "test_R2": r2_score(y_test, prediction),
    }
    for name, prediction in tuned_predictions.items()
}).T.round(3)

comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
comparison["test_MAE"].plot(kind="bar", ax=axes[0], color="#e76f51", rot=0)
axes[0].set_title("Lower test MAE is better")
axes[0].set_ylabel("MAE")

comparison["test_R2"].plot(kind="bar", ax=axes[1], color="#457b9d", rot=0)
axes[1].set_title("Higher test R2 is better")
axes[1].set_ylabel("R2")
axes[1].set_ylim(0, 1.05)

fig.suptitle("Tuned regularized-model comparison", weight="bold")
fig.tight_layout()
plt.show()

## 7. Final revision card

- Cross-validation chooses hyperparameters using training data only.
- LassoCV selects L1 alpha and can create zero coefficients.
- RidgeCV selects L2 alpha and generally keeps every feature.
- ElasticNetCV selects alpha and can choose the L1/L2 balance.
- CV=5 means five validation folds.
- alpha_ is the selected alpha; alphas_ shows the alpha values tried.
- mse_path_ records Lasso validation error across alphas and folds.
- Evaluate the selected model once on the untouched test data.

### One-line interview answer

**I use LassoCV, RidgeCV, or ElasticNetCV to choose regularization settings from training folds, then report final MAE and R2 on data that was never used for tuning.**